In [24]:
import pandas as pd
import glob

all_files = glob.glob("Product-Statistics-UPI*.xlsx")
print("Files found:", len(all_files))


Files found: 11


In [30]:
df_list = []
for file in all_files:
    df = pd.read_excel(file, sheet_name='Product Statistics', header=0)
    df_list.append(df)    

    df_raw = pd.concat(df_list, ignore_index=True)
    print("Total rows before cleaning:", len(df_raw))
    

Total rows before cleaning: 12
Total rows before cleaning: 24
Total rows before cleaning: 36
Total rows before cleaning: 48
Total rows before cleaning: 60
Total rows before cleaning: 72
Total rows before cleaning: 84
Total rows before cleaning: 96
Total rows before cleaning: 108
Total rows before cleaning: 120
Total rows before cleaning: 122


Files found: 11


In [32]:
## Rename Columns ##
df_raw.columns=['Month', 'Banks_Live', 'Volume_Mn', 'Value_Cr']

## Remove Bad rows where month is not a real date ##
df_raw = df_raw[df_raw['Month'].astype(str).str.contains('-', na=False)]
print("Rows after removing bad data:", len(df_raw))


Rows after removing bad data: 122


In [33]:
## Changing Data Types ##
df_raw['Volume_Mn'] = pd.to_numeric(df_raw['Volume_Mn'].astype(str).str.replace(',', ''), errors='coerce')
df_raw['Value_Cr'] = pd.to_numeric(df_raw['Value_Cr'].astype(str).str.replace(',', ''), errors='coerce')
df_raw['Banks_Live'] = pd.to_numeric(df_raw['Banks_Live'], errors='coerce')

## Converting to Month to date  ##
df_raw['Month'] = pd.to_datetime(df_raw['Month'], format='%B-%Y')

## Extract Date parts ##
df_raw['Year'] = df_raw['Month'].dt.year
df_raw['Month_Num'] = df_raw['Month'].dt.month
df_raw['Month_Name'] = df_raw['Month'].dt.strftime('%B')
df_raw['Quarter'] = df_raw['Month'].dt.quarter
df_raw['Financial_Year'] = df_raw['Month'].apply(
    lambda x: f"FY{x.year}-{str(x.year+1)[-2:]}" if x.month >= 4
    else f"FY{x.year-1}-{str(x.year)[-2:]}")

## Remove Duplicates ##
df_clean = df_raw.drop_duplicates(subset=['Month'])
df_clean = df_clean.sort_values('Month').reset_index(drop=True)


print("\n✅ Clean Data Shape:", df_clean.shape)
print("\nNull Values:\n", df_clean.isnull().sum())
print("\nFirst 5 rows:\n", df_clean.head())
print("\nLast 5 rows:\n", df_clean.tail())


✅ Clean Data Shape: (122, 9)

Null Values:
 Month             0
Banks_Live        0
Volume_Mn         0
Value_Cr          0
Year              0
Month_Num         0
Month_Name        0
Quarter           0
Financial_Year    0
dtype: int64

First 5 rows:
        Month  Banks_Live  Volume_Mn  Value_Cr  Year  Month_Num Month_Name  \
0 2016-04-01          21       0.00      0.00  2016          4      April   
1 2016-05-01          21       0.00      0.00  2016          5        May   
2 2016-06-01          21       0.00      0.00  2016          6       June   
3 2016-07-01          21       0.09      0.38  2016          7       July   
4 2016-08-01          21       0.09      3.09  2016          8     August   

   Quarter Financial_Year  
0        2      FY2016-17  
1        2      FY2016-17  
2        2      FY2016-17  
3        3      FY2016-17  
4        3      FY2016-17  

Last 5 rows:
          Month  Banks_Live  Volume_Mn    Value_Cr  Year  Month_Num Month_Name  \
117 2026-01-01     

In [34]:
df_clean.to_csv("upi_clean.csv", index=False)
print("✅ File saved as upi_clean.csv!")
print("Total rows saved:", len(df_clean))

✅ File saved as upi_clean.csv!
Total rows saved: 122
